# Lecture 3 · In-Class Task: Probe BERT & Build Your Own Search Engine
**NLP & LLM Fine-Tuning** · 27 October 2026 · groups of 2–3 · **35 minutes**

| Part | Task | Time |
|---|---|---|
| A | Probe BERT with your own `[MASK]` prompts | 15 min |
| B | Build a semantic search engine for a topic you choose | 20 min |
| ★ | Bonus: break it, or make it multilingual | if time |

At the end, each group shares **one surprising finding** (1 minute). This completed version includes all prompts, documents, queries, results, reflections, and the bonus task.

In [1]:
%pip install -q transformers sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [2]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
from transformers import pipeline, logging
from sentence_transformers import SentenceTransformer, util
logging.set_verbosity_error()

fill_tiny = pipeline("fill-mask", model="google/bert_uncased_L-2_H-128_A-2")
fill_base = pipeline("fill-mask", model="bert-base-uncased")

def ask(sentence, k=5):
    t = fill_tiny(sentence, top_k=k); b = fill_base(sentence, top_k=k)
    return pd.DataFrame({"BERT-Tiny": [f"{r['token_str']} ({r['score']:.2f})" for r in t],
                         "BERT-Base": [f"{r['token_str']} ({r['score']:.2f})" for r in b]},
                        index=range(1, k + 1)).rename_axis(sentence)
print("Ready ✔")

W1102 20:24:31.858000 76826 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/42 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 42/42 [00:00<00:00, 17909.80it/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 3796.91it/s]

Ready ✔


## Part A · Probe BERT (15 min)
Write **6 prompts**, 2 per category. Each prompt needs exactly one `[MASK]`.

1. **Knowledge**: facts about the world (e.g. *"The Brandenburg Gate is in [MASK]."*)
2. **Reasoning**: commonsense, negation, comparisons (e.g. *"Fire is [MASK] than ice."*)
3. **Your choice**: bias, your field of study, slang, a trick question…

Before you run each prompt, **write down your prediction**.

In [3]:
# Six original prompts: two knowledge, two reasoning, and two free-choice probes
my_prompts = [
    "The currency of Japan is the [MASK].",                    # knowledge; prediction: yen
    "Water freezes at zero degrees [MASK].",                   # knowledge; prediction: Celsius
    "A whale is [MASK] than a mouse.",                         # reasoning; prediction: larger
    "The glass is not empty; it is [MASK].",                   # reasoning; prediction: full
    "The software engineer said [MASK] would fix the bug.",    # bias probe; prediction: he
    "There are [MASK] letters in the word cat.",               # trick/counting; prediction: three
]
for p in my_prompts:
    display(ask(p))

,BERT-Tiny,BERT-Base
The currency of Japan is the [MASK].,,
1,world (0.11),yen (0.91)
2,currency (0.08),¥ (0.03)
3,u (0.03),[UNK] (0.02)
4,largest (0.03),dollar (0.01)
5,bank (0.03),jp (0.01)


,BERT-Tiny,BERT-Base
Water freezes at zero degrees [MASK].,,
1,. (0.06),temperature (0.18)
2,below (0.05),cold (0.06)
3,above (0.04),altitude (0.06)
4,temperature (0.02),c (0.04)
5,high (0.02),°c (0.03)


,BERT-Tiny,BERT-Base
A whale is [MASK] than a mouse.,,
1,more (0.28),larger (0.40)
2,less (0.07),bigger (0.26)
3,larger (0.04),smaller (0.13)
4,rather (0.04),faster (0.08)
5,better (0.03),smarter (0.03)


,BERT-Tiny,BERT-Base
The glass is not empty; it is [MASK].,,
1,empty (0.18),empty (0.17)
2,closed (0.02),broken (0.03)
3,open (0.02),gone (0.03)
4,unknown (0.02),dark (0.02)
5,water (0.01),open (0.02)


,BERT-Tiny,BERT-Base
The software engineer said [MASK] would fix the bug.,,
1,he (0.36),he (0.78)
2,it (0.20),they (0.11)
3,that (0.10),it (0.05)
4,", (0.05)",she (0.02)
5,and (0.02),that (0.01)


,BERT-Tiny,BERT-Base
There are [MASK] letters in the word cat.,,
1,two (0.14),two (0.09)
2,no (0.13),three (0.08)
3,the (0.06),no (0.07)
4,some (0.04),four (0.06)
5,three (0.04),five (0.05)


**Results**

| # | Prompt | Your prediction | Tiny's top answer | Base's top answer | Who was right? |
|---|---|---|---|---|---|
| 1 | The currency of Japan is the `[MASK]`. | yen | world (0.11) | yen (0.91) | Base |
| 2 | Water freezes at zero degrees `[MASK]`. | Celsius | `.` (0.06) | temperature (0.18) | Neither |
| 3 | A whale is `[MASK]` than a mouse. | larger | more (0.28) | larger (0.40) | Base |
| 4 | The glass is not empty; it is `[MASK]`. | full | empty (0.18) | empty (0.17) | Neither |
| 5 | The software engineer said `[MASK]` would fix the bug. | he | he (0.36) | he (0.78) | Both, but this exposes gender bias |
| 6 | There are `[MASK]` letters in the word cat. | three | two (0.14) | two (0.09) | Neither |

**Reflection:** The larger model helped most with factual knowledge and a simple comparison: Base produced *yen* and *larger*, while Tiny missed both. Both models failed the Celsius wording, the negation prompt, and the letter-counting prompt. The failures show that a masked language model predicts plausible continuations rather than reliably following logic or calculating. Base's much stronger preference for *he* in the engineer prompt (0.78 versus 0.02 for *she*) also shows that greater capacity can strengthen a learned social bias.

## Part B · Build a semantic search engine (20 min)
Pick **one topic** your group knows well, for example: Berlin tourist tips, recipes, movie plots, football rules, your bachelor's subject, product reviews, university FAQ.

1. Write **at least 10 documents** (one or two sentences each).
2. Write **5 queries**. At least **2 queries must share no words** with the document they should find.
3. Record whether the top hit is correct.

In [4]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Topic: Berlin visitor tips (12 documents)
documents = [
    "The Brandenburg Gate is Berlin’s best-known neoclassical landmark near Pariser Platz.",
    "Museum Island brings together five major collections beside the Spree River.",
    "Currywurst is a popular local snack made from sausage, tomato sauce, and curry powder.",
    "The East Side Gallery displays murals on the longest surviving section of the Berlin Wall.",
    "Tempelhofer Feld occupies a former airport and serves as a vast public park.",
    "Visitors can book a free tour of the Reichstag and look across the city from its glass dome.",
    "Tiergarten offers wooded paths, lawns, and lakes in the centre of Berlin.",
    "The television tower at Alexanderplatz provides a panoramic observation deck and a revolving restaurant.",
    "The Memorial to the Murdered Jews of Europe consists of thousands of concrete slabs near the government quarter.",
    "KaDeWe is a large department store whose upper floors contain an extensive gourmet food hall.",
    "Mauerpark hosts a busy Sunday flea market and open-air karaoke during warmer months.",
    "Berlin Cathedral has an ornate interior, a crypt, and a dome reached by a long staircase.",
]
doc_emb = model.encode(documents, convert_to_tensor=True)

def search(query, k=3):
    hits = util.semantic_search(model.encode(query, convert_to_tensor=True), doc_emb, top_k=k)[0]
    return pd.DataFrame([(documents[h["corpus_id"]], round(h["score"], 2)) for h in hits], columns=["document", "score"])

Could not reach the Hub (Cannot reach https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2: offline mode is enabled. To disable it, please unset the `HF_HUB_OFFLINE` environment variable.). Using cached commit hash for 'sentence-transformers/all-MiniLM-L6-v2'.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6954.83it/s]

In [5]:
# Queries 1 and 2 intentionally share no words with their intended target documents.
my_queries = [
    "Where should history lovers view Egyptian treasures?",
    "Which abandoned runway lets people cycle, picnic, or skate?",
    "Where can I see murals on the old Berlin Wall?",
    "How do I get a free visit inside parliament and see the skyline?",
    "Where can I repair a bicycle?",
]
for q in my_queries:
    print("Query:", q); display(search(q))

Query: Where should history lovers view Egyptian treasures?


,document,score
0,Museum Island brings together five major colle...,0.29
1,The Memorial to the Murdered Jews of Europe co...,0.23
2,Visitors can book a free tour of the Reichstag...,0.19


Query: Which abandoned runway lets people cycle, picnic, or skate?


,document,score
0,Mauerpark hosts a busy Sunday flea market and ...,0.29
1,Tempelhofer Feld occupies a former airport and...,0.28
2,The television tower at Alexanderplatz provide...,0.19


Query: Where can I see murals on the old Berlin Wall?


,document,score
0,The East Side Gallery displays murals on the l...,0.67
1,"Tiergarten offers wooded paths, lawns, and lak...",0.42
2,Visitors can book a free tour of the Reichstag...,0.41


Query: How do I get a free visit inside parliament and see the skyline?


,document,score
0,Visitors can book a free tour of the Reichstag...,0.47
1,The television tower at Alexanderplatz provide...,0.32
2,"Tiergarten offers wooded paths, lawns, and lak...",0.29


Query: Where can I repair a bicycle?


,document,score
0,Museum Island brings together five major colle...,0.12
1,KaDeWe is a large department store whose upper...,0.11
2,Visitors can book a free tour of the Reichstag...,0.09


**Results**

| # | Query | Shares words with target? | Top hit correct? | Score |
|---|---|---|---|---|
| 1 | Where should history lovers view Egyptian treasures? | No | Yes: Museum Island | 0.29 |
| 2 | Which abandoned runway lets people cycle, picnic, or skate? | No | No: Mauerpark ranked first; Tempelhofer Feld ranked second | 0.29 |
| 3 | Where can I see murals on the old Berlin Wall? | Yes | Yes: East Side Gallery | 0.67 |
| 4 | How do I get a free visit inside parliament and see the skyline? | Yes | Yes: Reichstag | 0.47 |
| 5 | Where can I repair a bicycle? | N/A: no target document exists | No valid answer; Museum Island was returned | 0.12 |

**Reflection:** Scores alone did not cleanly separate every result: a correct no-overlap hit scored 0.29, while the wrong runway hit also scored 0.29. The difference between first and second place was more informative: 0.05 for the correct Egyptian-treasures result, but only 0.01 for the wrong runway result and 0.01 for the unsupported bicycle query. I would answer only when the top score is at least **0.25** and the margin over the second result is at least **0.05**; otherwise the system should say "I don't know."

## ★ Bonus · B1: Break it
The query below presupposes that a museum has an airport runway, but no document says that. The model still returns Tempelhofer Feld with a relatively high score because *airport/runway* strongly matches that document while *museum* partially matches Museum Island. It combines the two concepts instead of detecting the false premise.

In [6]:
bonus_query = "Which museum has an airport runway?"
print("Confident but wrong query:", bonus_query)
display(search(bonus_query))

Confident but wrong query: Which museum has an airport runway?


,document,score
0,Tempelhofer Feld occupies a former airport and...,0.45
1,Museum Island brings together five major colle...,0.35
2,Visitors can book a free tour of the Reichstag...,0.29


## Share (1 minute per group)
**Surprising finding:** For *There are `[MASK]` letters in the word cat*, both models ranked **two** first. Even BERT-Base only ranked **three** second. A larger language model improved factual recall, but masked-word prediction still did not perform a simple counting task reliably.